### Title: 01_build_nhanes_wweia
### Purpose: Build out the WWEIA datasets by applying exclusion criteria, intergrating covariates and outcomes (insulin resistance [HOMA-IR]), adjusting intakes per 1000 kcal and matching participants across datasets (Mixed Meals, FPED, Nutrients, Ingredients)
### Date: July 23, 2025
### Author: Jules Larke

### Import packages

In [1]:
import pandas as pd
import numpy as np
import os

### Load data for the Ingredients dataset

In [2]:
# Load ingredients data
wweia_ingredients = pd.read_csv('../../data/00/wweia_dataset/wweia_ingredients_recalls_2023.csv')

### Apply the exclusion criteria for this dataset. The other datasets (mixed meals, nutrients, FPED) will be matched for participants at the end of the script.

In [3]:
# Make a copy of original data
wweia_ingredients = wweia_ingredients.copy()

# first, rename columns so they are more readable
wweia_ingredients.rename(columns={
    'RIDAGEYR': 'Age', 
    'RIAGENDR': 'Sex', 
    'RIDRETH1': 'ethnicity', 
    'INDFMPIR': 'family_pir',
    'DMDEDUC2': 'education'
}, inplace=True)

# subset to exclude participants  <20 years old
wweia_ingredients = wweia_ingredients[wweia_ingredients['Age'] > 20]

# factorize education levels
wweia_ingredients['education'] = wweia_ingredients['education'].replace([1, 2], 'less than high school graduate')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(3, 'high school graduate or equivalent')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(4, 'some college')
wweia_ingredients['education'] = wweia_ingredients['education'].replace(5, 'college graduate')
wweia_ingredients['education'] = wweia_ingredients['education'].replace([7, 9], 'unknown')

# remove education == unknown
wweia_ingredients = wweia_ingredients[wweia_ingredients['education']!='unknown']

# remove education variable for participants under 20, (excluded)
wweia_ingredients.drop(columns='DMDEDUC3', inplace=True)

# remove participants that are pregnant and drop column
wweia_ingredients = wweia_ingredients[wweia_ingredients['RIDEXPRG']!=1]
wweia_ingredients.drop(columns='RIDEXPRG', inplace=True)

In [4]:
# remove columns we won't be using (sample weights)
wweia_ingredients.drop(columns=['WTINT2YR', 'WTMEC2YR'], inplace=True)

### The remaining features and target (insulin resistance) need to be downloaded from the web. We will download those data and merge with the participant IDs (SEQN)

### Body Mass Index and Waist Circumference

In [5]:
# Exam data - Body Measures (BMI: BMXBMI, WC: BMXWAIST)
bmx_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/BMX_C.XPT', format='xport', encoding='utf-8')
bmx_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BMX_D.XPT', format='xport', encoding='utf-8')
bmx_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/BMX_E.XPT', format='xport', encoding='utf-8')
bmx_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BMX_F.XPT', format='xport', encoding='utf-8')
bmx_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BMX_G.XPT', format='xport', encoding='utf-8')
bmx_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BMX_H.XPT', format='xport', encoding='utf-8')
bmx_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BMX_I.XPT', format='xport', encoding='utf-8')
bmx_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BMX.xpt', format='xport', encoding='utf-8')
bmx_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BMX_L.xpt', format='xport', encoding='utf-8')

# select features (BMI, WC)
bmx_C = bmx_C[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_D = bmx_D[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_E = bmx_E[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_F = bmx_F[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_G = bmx_G[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_H = bmx_H[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_I = bmx_I[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_P = bmx_P[['SEQN', 'BMXBMI', 'BMXWAIST']]
bmx_L = bmx_L[['SEQN', 'BMXBMI', 'BMXWAIST']]

bmi = pd.concat([bmx_C, bmx_D, bmx_E, bmx_F, bmx_G, bmx_H, bmx_I, bmx_P, bmx_L])
bmi.rename(columns={'BMXBMI': 'BMI', 'BMXWAIST': 'WC'}, inplace=True)

wweia_ingredients = wweia_ingredients.merge(bmi, on='SEQN', how='left')

### Fasting Glucose and Insulin

In [6]:
#Lab data - fasting glucose

fg_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L10AM_C.XPT', format='xport', encoding='utf-8')
fg_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/GLU_D.XPT', format='xport', encoding='utf-8')
fg_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/GLU_E.XPT', format='xport', encoding='utf-8')
fg_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/GLU_F.XPT', format='xport', encoding='utf-8')
fg_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/GLU_G.XPT', format='xport', encoding='utf-8')
fg_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/GLU_H.XPT', format='xport', encoding='utf-8')
fg_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/GLU_I.XPT', format='xport', encoding='utf-8')
fg_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GLU.XPT', format='xport', encoding='utf-8')
fg_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GLU_L.XPT', format='xport', encoding='utf-8')

# insulin data for last years
ins_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/INS_H.xpt', format='xport', encoding='utf-8')
ins_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/INS_I.xpt', format='xport', encoding='utf-8')
ins_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_INS.xpt', format='xport', encoding='utf-8')
ins_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/INS_L.xpt', format='xport', encoding='utf-8')
ins_H = ins_H[['SEQN', 'LBXIN']]
ins_I = ins_I[['SEQN', 'LBXIN']]
ins_P = ins_P[['SEQN', 'LBXIN']]
ins_L = ins_L[['SEQN', 'LBXIN']]

# remove those with a sample weight (WTSAF2YR) of zero for fasting glucose, indicates fasting was not completed by the participant
fg_C = fg_C[['SEQN', 'LBXGLU', 'LBXIN', 'WTSAF2YR']]
fg_C = fg_C[fg_C['WTSAF2YR']<1]
fg_C = fg_C.drop(columns='WTSAF2YR')

fg_I = fg_I[['SEQN', 'LBXGLU', 'WTSAF2YR']]
fg_P = fg_P[['SEQN', 'LBXGLU', 'WTSAFPRP']]
fg_L = fg_L[['SEQN', 'LBXGLU', 'WTSAF2YR']]
fg_I = fg_I[fg_I['WTSAF2YR']>1]
fg_P = fg_P[fg_P['WTSAFPRP']>1]
fg_L = fg_L[fg_L['WTSAF2YR']>1]

fg_H = fg_H.merge(ins_H, on='SEQN')
fg_I = fg_I.merge(ins_I, on='SEQN')
fg_P = fg_P.merge(ins_P, on='SEQN')
fg_L = fg_L.merge(ins_L, on='SEQN')

# fasting time was included in these cycles (PHAFSTHR)
fg_D = fg_D[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_E = fg_E[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_F = fg_F[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_G = fg_G[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]
fg_H = fg_H[['SEQN', 'LBXGLU', 'LBXIN', 'PHAFSTHR']]

# remove participants with fasting time of less than 8 hours
fg_D = fg_D[fg_D['PHAFSTHR']>=8]
fg_E = fg_E[fg_E['PHAFSTHR']>=8]
fg_F = fg_F[fg_F['PHAFSTHR']>=8]
fg_G = fg_G[fg_G['PHAFSTHR']>=8]
fg_H = fg_H[fg_H['PHAFSTHR']>=8]

# select features for cycles
fg_C = fg_C[['SEQN', 'LBXGLU', 'LBXIN']]
fg_D = fg_D[['SEQN', 'LBXGLU', 'LBXIN']]
fg_E = fg_E[['SEQN', 'LBXGLU', 'LBXIN']]
fg_F = fg_F[['SEQN', 'LBXGLU', 'LBXIN']]
fg_G = fg_G[['SEQN', 'LBXGLU', 'LBXIN']]
fg_H = fg_H[['SEQN', 'LBXGLU', 'LBXIN']]
fg_I = fg_I[['SEQN', 'LBXGLU', 'LBXIN']]
fg_P = fg_P[['SEQN', 'LBXGLU', 'LBXIN']]
fg_L = fg_L[['SEQN', 'LBXGLU', 'LBXIN']]

fg = pd.concat([fg_C, fg_D, fg_E, fg_F, fg_G, fg_H, fg_I, fg_P, fg_L])
fg.rename(columns={'LBXGLU':'fasting_glc_mg_dL', 'LBXIN': 'fasting_ins_uU_mL'}, inplace=True)

### Calibration of fasting glucose and insulin
#### Due to changes in instrumentation over NHANES cycles, equations are provided by CDC (NHANES) to adjust values and compare cycles

In [7]:
# get the unique participants and their NHANES cycle 
seqn_cycle = wweia_ingredients.drop_duplicates(subset='SEQN')[['SEQN', 'CYCLE']]

# merge the fasting glucose and insulin data with participant cycle data
fg_cycle = fg.merge(seqn_cycle, on='SEQN', how='left')

# drop NAs
fg_cycle = fg_cycle.dropna()

In [8]:
# function to apply calibrations to the cycle years. This will chain the equations for each cycle where an instrument change occured, successively applying the corrections
# a reference standard was selected for when the last instrument change occurred

def calibrate_nhanes_glucose_insulin(df):
    """
    Calibrate NHANES glucose and insulin measurements from 2003-2023.
    
    GLUCOSE: Uses 2015-2016 (C311) as reference standard
    INSULIN: Uses 2013-2014+ (Tosoh AIA) as reference standard
    
    Parameters:
    df: DataFrame with columns ['SEQN', 'fasting_glc_mg_dL', 'fasting_ins_uU_mL', 'CYCLE']
    
    Returns:
    DataFrame with calibrated glucose and insulin values
    """
    
    df_calibrated = df.copy()
    df_calibrated['glucose_calibrated'] = df_calibrated['fasting_glc_mg_dL'].copy()
    df_calibrated['insulin_calibrated'] = df_calibrated['fasting_ins_uU_mL'].copy()
    
    # =============================================================================
    # GLUCOSE CALIBRATION (Reference: 2015-2016 C311)
    # =============================================================================
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        glucose_orig = row['fasting_glc_mg_dL']
        
        if pd.isna(glucose_orig):
            continue
            
        glucose_cal = glucose_orig
        
        if cycle == '03_04':
            # 2003-2004 (Cobas Mira) → 2015-2016 (C311 reference)
            # Step 1: Mira → 911: Y(911) = 0.9815 * X(Mira) + 3.5707
            glucose_cal = 0.9815 * glucose_cal + 3.5707
            
            # Step 2: 911 → ModP: Y(ModP) = X(911) + 1.148
            glucose_cal = glucose_cal + 1.148
            
            # Step 3: ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle == '05_06':
            # 2005-2006 (Hitachi 911) → 2015-2016 (C311 reference)
            # Step 1: 911 → ModP: Y(ModP) = X(911) + 1.148
            glucose_cal = glucose_cal + 1.148
            
            # Step 2: ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle in ['07_08', '09_10', '11_12', '13_14']:
            # 2007-2014 (ModP/C501) → 2015-2016 (C311 reference)
            # ModP/C501 → C311: X(C311) = (Y(C501) - 0.4994) / 0.9776
            glucose_cal = (glucose_cal - 0.4994) / 0.9776
            
        elif cycle in ['15_16', '17_20', '21_22']:
            # 2015-2016+: Same C311 instrument - NO calibration needed
            glucose_cal = glucose_cal
            
        df_calibrated.loc[idx, 'glucose_calibrated'] = glucose_cal
    
    # =============================================================================
    # INSULIN CALIBRATION (Reference: 2013-2014+ Tosoh AIA)
    # =============================================================================
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        insulin_orig = row['fasting_ins_uU_mL']
        
        if pd.isna(insulin_orig) or insulin_orig <= 0:
            continue
            
        insulin_cal = insulin_orig
        
        if cycle == '03_04':
            # 2003-2004 (Tosoh) → 2013-2014+ (Tosoh AIA reference)
            # Step 1: 2003-04 Tosoh → Mercodia
            # Y(Mercodia-equivalent) = 0.9501*X(TOSOH) + 1.4890
            insulin_cal = 0.9501 * insulin_cal + 1.4890
            
            # Step 2: Mercodia → Roche
            # Insulin (Roche-equivalent) = 0.8868*Insulin (Mercodia) + [0.0011*Insulin (Mercodia)**2] – 0.0744
            insulin_cal = 0.8868 * insulin_cal + 0.0011 * (insulin_cal ** 2) - 0.0744
            
            # Step 3: Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['05_06', '07_08']:
            # 2005-2008 (Mercodia) → 2013-2014+ (Tosoh AIA reference)
            # Step 1: Mercodia → Roche
            # Insulin (Roche-equivalent) = 0.8868*Insulin (Mercodia) + [0.0011*Insulin (Mercodia)**2] – 0.0744
            insulin_cal = 0.8868 * insulin_cal + 0.0011 * (insulin_cal ** 2) - 0.0744
            
            # Step 2: Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['09_10', '11_12']:
            # 2009-2012 (Roche) → 2013-2014+ (Tosoh AIA reference)
            # Roche → Tosoh AIA
            # Insulin (Tosoh-equivalent) = 10**(1.024*log10(Roche insulin) – 0.0802)
            if insulin_cal > 0:
                insulin_cal = 10 ** (1.024 * np.log10(insulin_cal) - 0.0802)
            
        elif cycle in ['13_14', '15_16', '17_20', '21_22']:
            # 2013-2014+: Tosoh AIA - reference standard, no calibration needed
            insulin_cal = insulin_cal
            
        df_calibrated.loc[idx, 'insulin_calibrated'] = insulin_cal
    
    return df_calibrated

def add_calibration_notes(df):
    """Add detailed notes about calibration equations applied"""
    
    calibration_notes = {
        '03_04': 'Glucose: Mira→911→ModP→C311 | Insulin: Tosoh→Mercodia→Roche→Tosoh AIA',
        '05_06': 'Glucose: 911→ModP→C311 | Insulin: Mercodia→Roche→Tosoh AIA', 
        '07_08': 'Glucose: ModP→C311 | Insulin: Mercodia→Roche→Tosoh AIA',
        '09_10': 'Glucose: ModP→C311 | Insulin: Roche→Tosoh AIA',
        '11_12': 'Glucose: ModP→C311 | Insulin: Roche→Tosoh AIA',
        '13_14': 'Glucose: C501→C311 | Insulin: REFERENCE (Tosoh AIA)',
        '15_16': 'Glucose: REFERENCE (C311) | Insulin: REFERENCE (Tosoh AIA)',
        '17_20': 'Glucose: NO CALIBRATION (same C311) | Insulin: REFERENCE (Tosoh AIA)', 
        '21_22': 'Glucose: NO CALIBRATION (same C311) | Insulin: REFERENCE (Tosoh AIA)'
    }
    
    df['calibration_equations'] = df['CYCLE'].map(calibration_notes)
    return df

def print_calibration_summary():
    """Print summary of complete calibration approach"""
    
    print("COMPLETE NHANES GLUCOSE & INSULIN CALIBRATION")
    print("=" * 70)
    print("\nGLUCOSE REFERENCE: 2015-2016 (C311)")
    print("Timeline: Mira → 911 → ModP/C501 → C311")
    print("  Mira→911: Y(911) = 0.9815 * X(Mira) + 3.5707")
    print("  911→ModP: Y(ModP) = X(911) + 1.148")
    print("  C501→C311 inverse: X(C311) = (Y(C501) - 0.4994) / 0.9776")
    print("\nINSULIN REFERENCE: 2013-2014+ (Tosoh AIA)")
    print("Timeline: 2003 Tosoh → Mercodia → Roche → Tosoh AIA")
    print("  2003 Tosoh→Mercodia: Y(Mercodia) = 0.9501*X(Tosoh) + 1.4890")
    print("  Mercodia→Roche: Y(Roche) = 0.8868*X(Mercodia) + 0.0011*X²- 0.0744")
    print("  Roche→Tosoh AIA: Y(Tosoh) = 10**(1.024*log10(X) - 0.0802)")
    print("\nINSULIN METHOD TIMELINE:")
    print("  2003-04: Tosoh method")
    print("  2005-08: Mercodia ELISA")
    print("  2009-12: Roche Elecsys")
    print("  2013+:   Tosoh AIA (reference)")
    print("\nCALIBRATION CHAINS:")
    print("  2003-04: Tosoh→Mercodia→Roche→Tosoh AIA (3 steps)")
    print("  2005-08: Mercodia→Roche→Tosoh AIA (2 steps)")
    print("  2009-12: Roche→Tosoh AIA (1 step)")
    print("  2013+:   Reference (no calibration)")
    print("=" * 70)

# Usage function to apply calibrations to your dataframe
def apply_calibrations(df):
    """
    Apply calibrations to your NHANES dataframe
    
    Parameters:
    df: Your dataframe with columns ['SEQN', 'fasting_glc_mg_dL', 'fasting_ins_uU_mL', 'CYCLE']
    
    Returns:
    Calibrated dataframe saved as glu_ins_cal
    """
    
    # Print calibration summary
    print_calibration_summary()
    print("\n")
    
    # Apply calibrations
    df_calibrated = calibrate_nhanes_glucose_insulin(df)
    df_calibrated = add_calibration_notes(df_calibrated)
    
    # Calculate changes for reference
    df_calibrated['glucose_change'] = (df_calibrated['glucose_calibrated'] - 
                                      df_calibrated['fasting_glc_mg_dL']).round(2)
    
    df_calibrated['insulin_change'] = (df_calibrated['insulin_calibrated'] - 
                                      df_calibrated['fasting_ins_uU_mL']).round(2)
    
    # Save calibrated results to specified dataframe name
    global glu_ins_cal
    glu_ins_cal = df_calibrated
    
    print("CALIBRATION COMPLETE!")
    print(f"Calibrated results saved to 'glu_ins_cal' dataframe")
    print(f"Shape: {glu_ins_cal.shape}")
    print("\nNew columns added:")
    print("  - glucose_calibrated: Calibrated glucose values")
    print("  - insulin_calibrated: Calibrated insulin values") 
    print("  - calibration_equations: Notes on equations applied")
    print("  - glucose_change: Change from original glucose")
    print("  - insulin_change: Change from original insulin")
    
    return glu_ins_cal

In [9]:
# apply the calibrations to the data
glu_ins_cal = apply_calibrations(fg_cycle)

COMPLETE NHANES GLUCOSE & INSULIN CALIBRATION

GLUCOSE REFERENCE: 2015-2016 (C311)
Timeline: Mira → 911 → ModP/C501 → C311
  Mira→911: Y(911) = 0.9815 * X(Mira) + 3.5707
  911→ModP: Y(ModP) = X(911) + 1.148
  C501→C311 inverse: X(C311) = (Y(C501) - 0.4994) / 0.9776

INSULIN REFERENCE: 2013-2014+ (Tosoh AIA)
Timeline: 2003 Tosoh → Mercodia → Roche → Tosoh AIA
  2003 Tosoh→Mercodia: Y(Mercodia) = 0.9501*X(Tosoh) + 1.4890
  Mercodia→Roche: Y(Roche) = 0.8868*X(Mercodia) + 0.0011*X²- 0.0744
  Roche→Tosoh AIA: Y(Tosoh) = 10**(1.024*log10(X) - 0.0802)

INSULIN METHOD TIMELINE:
  2003-04: Tosoh method
  2005-08: Mercodia ELISA
  2009-12: Roche Elecsys
  2013+:   Tosoh AIA (reference)

CALIBRATION CHAINS:
  2003-04: Tosoh→Mercodia→Roche→Tosoh AIA (3 steps)
  2005-08: Mercodia→Roche→Tosoh AIA (2 steps)
  2009-12: Roche→Tosoh AIA (1 step)
  2013+:   Reference (no calibration)


CALIBRATION COMPLETE!
Calibrated results saved to 'glu_ins_cal' dataframe
Shape: (16188, 9)

New columns added:
  - gluc

In [10]:
# select the calibrated values for fasting glucose and insulin
glu_ins_res = glu_ins_cal[['SEQN', 'glucose_calibrated', 'insulin_calibrated']]

# merge the calibrated data with the ingredients dataset
wweia_ingredients = wweia_ingredients.merge(glu_ins_res, on='SEQN', how='left')

# create variable to designate diabetic status based on fasting glucose
wweia_ingredients['diabetes_fasting_glc'] = np.where(wweia_ingredients['glucose_calibrated'] >= 126, 'yes', 'no')

### HbA1c (glycated hemoglobin)

In [11]:
#Lab data - glycohemoglobic (hba1c)

gh_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L10_C.XPT', format='xport', encoding='utf-8')
gh_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/GHB_D.XPT', format='xport', encoding='utf-8')
gh_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/GHB_E.XPT', format='xport', encoding='utf-8')
gh_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/GHB_F.XPT', format='xport', encoding='utf-8')
gh_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/GHB_G.XPT', format='xport', encoding='utf-8')
gh_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/GHB_H.XPT', format='xport', encoding='utf-8')
gh_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/GHB_I.XPT', format='xport', encoding='utf-8')
gh_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_GHB.xpt', format='xport', encoding='utf-8')
gh_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/GHB_L.xpt', format='xport', encoding='utf-8')

# select feature
gh_C = gh_C[['SEQN', 'LBXGH']]
gh_D = gh_D[['SEQN', 'LBXGH']]
gh_E = gh_E[['SEQN', 'LBXGH']]
gh_F = gh_F[['SEQN', 'LBXGH']]
gh_G = gh_G[['SEQN', 'LBXGH']]
gh_H = gh_H[['SEQN', 'LBXGH']]
gh_I = gh_I[['SEQN', 'LBXGH']]
gh_P = gh_P[['SEQN', 'LBXGH']]
gh_L = gh_L[['SEQN', 'LBXGH']]

gh = pd.concat([gh_C, gh_D, gh_E, gh_F, gh_G, gh_H, gh_I, gh_P, gh_L])
gh.rename(columns={'LBXGH':'hba1c_percent'}, inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(gh, on='SEQN', how='left')

# create variable to designate diabetic status based on hba1c clinical cutoff
wweia_ingredients['diabetes_hba1c'] = np.where(wweia_ingredients['hba1c_percent'] >= 6.5, 'yes', 'no')

### Questionnaire for taking medications that affect blood glucose

In [12]:
#Questionnaire data - taking insulin or glucose lowering meds

diq_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/DIQ_C.XPT', format='xport', encoding='utf-8')
diq_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/DIQ_D.XPT', format='xport', encoding='utf-8')
diq_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/DIQ_E.XPT', format='xport', encoding='utf-8')
diq_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/DIQ_F.XPT', format='xport', encoding='utf-8')
diq_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DIQ_G.XPT', format='xport', encoding='utf-8')
diq_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/DIQ_H.XPT', format='xport', encoding='utf-8')
diq_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/DIQ_I.XPT', format='xport', encoding='utf-8')
diq_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DIQ.xpt', format='xport', encoding='utf-8')
diq_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DIQ_L.xpt', format='xport', encoding='utf-8')

# select features
diq_C = diq_C[['SEQN', 'DIQ050', 'DIQ070']]
diq_D = diq_D[['SEQN', 'DIQ050', 'DID070']]
diq_E = diq_E[['SEQN', 'DIQ050', 'DID070']]
diq_F = diq_F[['SEQN', 'DIQ050', 'DIQ070']]
diq_G = diq_G[['SEQN', 'DIQ050', 'DIQ070']]
diq_H = diq_H[['SEQN', 'DIQ050', 'DIQ070']]
diq_I = diq_I[['SEQN', 'DIQ050', 'DIQ070']]
diq_P = diq_P[['SEQN', 'DIQ050', 'DIQ070']]
diq_L = diq_L[['SEQN', 'DIQ050', 'DIQ070']]

# hamonize names (same question)
diq_D.rename(columns={'DID070':'DIQ070'}, inplace=True)
diq_E.rename(columns={'DID070':'DIQ070'}, inplace=True)

diq = pd.concat([diq_C, diq_D, diq_E, diq_F, diq_G, diq_H, diq_I, diq_P, diq_L])
diq.rename(columns={'DIQ050':'taking_insulin', 'DIQ070':'taking_diabetic_pills'}, inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(diq, on='SEQN', how='left')

# recode levels for taking insulin / diabetes meds
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace(1, 'yes')
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace(2, 'no')
wweia_ingredients['taking_insulin'] = wweia_ingredients['taking_insulin'].replace([7, 9], 'unknown')

wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace(1, 'yes')
wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace(2, 'no')
wweia_ingredients['taking_diabetic_pills'] = wweia_ingredients['taking_diabetic_pills'].replace([7, 9], 'unknown')

### Determine if participant is diabetic based on the following criteria:
1. Fasting blood glucose > 126 mg/dL
2. HbA1c > 6.5%
3. Reported taking insulin or other diabetes medication 

In [13]:
# create single feature for diabetes (yes/no)
wweia_ingredients['diabetes'] = np.where(wweia_ingredients['diabetes_fasting_glc'] == 'yes', 'yes', 'no')
wweia_ingredients['diabetes'] = np.where(wweia_ingredients['hba1c_percent'] == 'yes', 'yes', wweia_ingredients['diabetes'])
wweia_ingredients['diabetes'] = np.where(wweia_ingredients['taking_insulin'] == 'yes', 'yes', wweia_ingredients['diabetes'])
wweia_ingredients['diabetes'] = np.where(wweia_ingredients['taking_diabetic_pills'] == 'yes', 'yes', wweia_ingredients['diabetes'])

In [14]:
# remove participants that have diabetes according to above criteria to remove confounding based on the outcomes of interest: insulin resistance / systemic inflammation
wweia_ingredients = wweia_ingredients[wweia_ingredients['diabetes']=='no']

# clean up columns
wweia_ingredients.drop(columns=['diabetes_fasting_glc', 'hba1c_percent', 'taking_insulin', 'taking_diabetic_pills'], inplace=True)

### Hypertension

In [15]:
#Exam data - blood pressure (hypertension)

bp_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/BPX_C.XPT', format='xport', encoding='utf-8')
bp_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/BPX_D.XPT', format='xport', encoding='utf-8')
bp_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/BPX_E.XPT', format='xport', encoding='utf-8')
bp_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/BPX_F.XPT', format='xport', encoding='utf-8')
bp_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/BPX_G.XPT', format='xport', encoding='utf-8')
bp_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/BPX_H.XPT', format='xport', encoding='utf-8')
bp_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/BPX_I.XPT', format='xport', encoding='utf-8')
bp_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BPXO.XPT', format='xport', encoding='utf-8')
bp_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/BPXO_L.XPT', format='xport', encoding='utf-8')

## systolic
# removed the fourth systolic measurment for cycles B-I as cycles P-L only have three measurements
bp_sys_C = bp_C[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_D = bp_D[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_E = bp_E[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_F = bp_F[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_G = bp_G[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_H = bp_H[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_I = bp_I[['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3']]
bp_sys_P = bp_P[['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3']]
bp_sys_L = bp_L[['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3']]

# create copy
bp_sys_P = bp_sys_P.copy()
bp_sys_L = bp_sys_L.copy()

# harmonize column names
bp_sys_P.rename(columns={'BPXOSY1': 'BPXSY1', 'BPXOSY2': 'BPXSY2', 'BPXOSY3': 'BPXSY3'}, inplace=True)
bp_sys_L.rename(columns={'BPXOSY1': 'BPXSY1', 'BPXOSY2': 'BPXSY2', 'BPXOSY3': 'BPXSY3'}, inplace=True)

# join cycles
bp_sys = pd.concat([bp_sys_C, bp_sys_D, bp_sys_E, bp_sys_F, bp_sys_G, bp_sys_H, bp_sys_I, bp_sys_P, bp_sys_L])
bp_sys.set_index('SEQN', inplace=True)

# compute the arithmetic mean for all measurements ignoring NaNs
bp_sys['sys_mean'] = bp_sys.mean(axis=1, skipna=True)
bp_sys = bp_sys.reset_index()[['SEQN', 'sys_mean']]

# designate hypertensive if systolic BP > 140
bp_sys['sys_ht'] = np.where(bp_sys['sys_mean'] >= 140, 'yes', 'no')

# clean up columns
bp_sys.drop(columns='sys_mean', inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(bp_sys, on='SEQN', how='left')

## diastolic
# removed the fourth diastolic measurment for cycles B-I as cycles P-L only have three measurements
bp_di_C = bp_C[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_D = bp_D[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_E = bp_E[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_F = bp_F[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_G = bp_G[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_H = bp_H[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_I = bp_I[['SEQN', 'BPXDI1', 'BPXDI2', 'BPXDI3']]
bp_di_P = bp_P[['SEQN', 'BPXODI1', 'BPXODI2', 'BPXODI3']]
bp_di_L = bp_L[['SEQN', 'BPXODI1', 'BPXODI2', 'BPXODI3']]

# create copy
bp_di_P = bp_di_P.copy()
bp_di_L = bp_di_L.copy()

bp_sys_P.rename(columns={'BPXODI1': 'BPXDI1', 'BPXODI1': 'BPXDI2', 'BPXODI1': 'BPXDI3'}, inplace=True)
bp_sys_L.rename(columns={'BPXODI1': 'BPXDI3', 'BPXODI1': 'BPXDI3', 'BPXODI1': 'BPXDI3'}, inplace=True)


bp_di = pd.concat([bp_di_C, bp_di_D, bp_di_E, bp_di_F, bp_di_G, bp_di_H, bp_di_I, bp_di_P, bp_di_L])
bp_di.set_index('SEQN', inplace=True)


# compute the arithmetic mean for all measurements ignoring NaNs
bp_di['di_mean'] = bp_di.mean(axis=1, skipna=True)
bp_di = bp_di.reset_index()[['SEQN', 'di_mean']]

# designate hypertensive if diastolic BP > 90
bp_di['di_ht'] = np.where(bp_di['di_mean'] >= 90, 'yes', 'no')

# clean up columns
bp_di.drop(columns='di_mean', inplace=True)

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(bp_di, on='SEQN', how='left')

In [16]:
# create varible to consider participants hypertensive if BP is over clinical threshold (no questionnaire data used)
wweia_ingredients['hypertension'] = np.where(wweia_ingredients['sys_ht'] == 'yes', 'yes', 'no')
wweia_ingredients['hypertension'] = np.where(wweia_ingredients['di_ht'] == 'yes', 'yes', wweia_ingredients['hypertension'])

### CVD / Cancer

In [17]:
#Questionnaire data - CVD / Cancer

mc_B = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2001/DataFiles/MCQ_B.XPT', format='xport', encoding='utf-8')
mc_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/MCQ_C.XPT', format='xport', encoding='utf-8')
mc_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/MCQ_D.XPT', format='xport', encoding='utf-8')
mc_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/MCQ_E.XPT', format='xport', encoding='utf-8')
mc_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/MCQ_F.XPT', format='xport', encoding='utf-8')
mc_G = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/MCQ_G.XPT', format='xport', encoding='utf-8')
mc_H = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/MCQ_H.XPT', format='xport', encoding='utf-8')
mc_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/MCQ_I.XPT', format='xport', encoding='utf-8')
mc_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_MCQ.XPT', format='xport', encoding='utf-8')
mc_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/MCQ_L.XPT', format='xport', encoding='utf-8')

mc_B = mc_B[['SEQN', 'MCQ160C', 'MCQ220']]
mc_C = mc_C[['SEQN', 'MCQ160C', 'MCQ220']]
mc_D = mc_D[['SEQN', 'MCQ160C', 'MCQ220']]
mc_E = mc_E[['SEQN', 'MCQ160C', 'MCQ220']]
mc_F = mc_F[['SEQN', 'MCQ160C', 'MCQ220']]
mc_G = mc_G[['SEQN', 'MCQ160C', 'MCQ220']]
mc_H = mc_H[['SEQN', 'MCQ160C', 'MCQ220']]
mc_I = mc_I[['SEQN', 'MCQ160C', 'MCQ220']]
mc_P = mc_P[['SEQN', 'MCQ160C', 'MCQ220']]
mc_L = mc_L[['SEQN', 'MCQ160C', 'MCQ220']]

# concat the cycles and merge with the ingredients dataset
mc = pd.concat([mc_B, mc_C, mc_D, mc_E, mc_F, mc_G, mc_H, mc_I, mc_P, mc_L])
wweia_ingredients = wweia_ingredients.merge(mc, on='SEQN', how='left')

# recode 0/1 to no/yes
wweia_ingredients['MCQ160C'] = np.where(wweia_ingredients['MCQ160C'] == 1, 'yes', 'no')
wweia_ingredients['MCQ220'] = np.where(wweia_ingredients['MCQ220'] == 1, 'yes', 'no')

# remove if ever told had CHD
wweia_ingredients = wweia_ingredients[wweia_ingredients['MCQ160C'] != 'yes']

# remove if ever told had Cancer
wweia_ingredients = wweia_ingredients[wweia_ingredients['MCQ220'] != 'yes']

### C-reactive protein

In [18]:
#Lab data - CRP (No CRP data collected from 2011-2014)

crp_C = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2003/DataFiles/L11_C.XPT', format='xport', encoding='utf-8')
crp_D = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2005/DataFiles/CRP_D.XPT', format='xport', encoding='utf-8')
crp_E = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2007/DataFiles/CRP_E.XPT', format='xport', encoding='utf-8')
crp_F = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2009/DataFiles/CRP_F.XPT', format='xport', encoding='utf-8')
crp_I = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2015/DataFiles/HSCRP_I.XPT', format='xport', encoding='utf-8')
crp_P = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_HSCRP.XPT', format='xport', encoding='utf-8')
crp_L = pd.read_sas('https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/HSCRP_L.xpt', format='xport', encoding='utf-8')

crp_I.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_P.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_L.rename(columns={'LBXHSCRP':'LBXCRP'}, inplace=True)
crp_L = crp_L.drop(columns='WTPH2YR')

crp_C = crp_C[['SEQN', 'LBXCRP']]
crp_D = crp_D[['SEQN', 'LBXCRP']]
crp_E = crp_E[['SEQN', 'LBXCRP']]
crp_F = crp_F[['SEQN', 'LBXCRP']]
crp_I = crp_I[['SEQN', 'LBXCRP']]
crp_P = crp_P[['SEQN', 'LBXCRP']]
crp_L = crp_L[['SEQN', 'LBXCRP']]

# these cycles measured in units mg/dL, need to convert
crp_convert = pd.concat([crp_C, crp_D, crp_E, crp_F])

# convert from mg/dL to mg/L
crp_convert['LBXCRP'] = crp_convert['LBXCRP'] * 10 

# join converted data
crp = pd.concat([crp_convert, crp_I, crp_P, crp_L])

# drop NAs
crp = crp.dropna()

In [19]:
# rename CRP
crp.rename(columns={'LBXCRP':'crp'},inplace=True)

# merge the CRP data with participant cycle data
crp_cycle = crp.merge(seqn_cycle, on='SEQN', how='left')

# drop NAs
crp_cycle = crp_cycle.dropna()

#### Calibrate CRP data

In [20]:
# just as we have done with fasting glucose and insulin, we need to calibrate CRP data due to changes in instrumentation over NHANES cycles

def calibrate_nhanes_crp(df):
    """
    Calibrate NHANES C-reactive protein measurements from 2015-2023.
    
    CRP: Uses 2017-2020 (Cobas 6000) as reference standard
    
    Parameters:
    df: DataFrame with columns ['SEQN', 'crp', 'CYCLE']
    
    Returns:
    DataFrame with calibrated CRP values
    """
    
    df_calibrated = df.copy()
    df_calibrated['crp_calibrated'] = df_calibrated['crp'].copy()
    
    # =============================================================================
    # CRP CALIBRATION (Reference: 2017-2020 Cobas 6000)
    # =============================================================================
    
    # Instrument timeline:
    # 2015-16: DxC 660i → 2017-20: Cobas 6000 (reference) → 2021-23: Cobas 8000
    
    for idx, row in df_calibrated.iterrows():
        cycle = row['CYCLE']
        crp_orig = row['crp']
        
        if pd.isna(crp_orig):
            continue
            
        crp_cal = crp_orig
        
        if cycle == '15_16':
            # 2015-2016 (DxC 660i) → 2017-2020 (Cobas 6000 reference)
            # Given: Y(DxC 660i) = 1.150 * X(Cobas 6000) – 0.3397
            # Inverse: X(Cobas 6000) = (Y(DxC 660i) + 0.3397) / 1.150
            # Note: Equation only applicable for Cobas 6000 values ≤ 20 mg/L
            crp_cal = (crp_cal + 0.3397) / 1.150
            
        elif cycle == '17_20':
            # 2017-2020 (Cobas 6000): Reference standard - NO calibration needed
            crp_cal = crp_cal
            
        elif cycle == '21_22':
            # 2021-2023 (Cobas 8000) → 2017-2020 (Cobas 6000 reference)
            # Given: Forward: Cobas 8000 = -0.08653 + 0.9672 * (Cobas 6000)
            # Inverse: Cobas 6000 = (Cobas 8000 + 0.08653) / 0.9672
            crp_cal = (crp_cal + 0.08653) / 0.9672
            
        df_calibrated.loc[idx, 'crp_calibrated'] = crp_cal
    
    return df_calibrated

def add_crp_calibration_notes(df):
    """Add detailed notes about CRP calibration equations applied"""
    
    calibration_notes = {
        '15_16': 'CRP: DxC 660i→Cobas 6000: (x + 0.3397) / 1.150 [≤20 mg/L only]',
        '17_20': 'CRP: REFERENCE (Cobas 6000)',
        '21_22': 'CRP: Cobas 8000→Cobas 6000: (x + 0.08653) / 0.9672'
    }
    
    df['crp_calibration_equations'] = df['CYCLE'].map(calibration_notes)
    return df

def print_crp_calibration_summary():
    """Print summary of CRP calibration approach"""
    
    print("NHANES C-REACTIVE PROTEIN CALIBRATION")
    print("=" * 60)
    print("\nCRP REFERENCE: 2017-2020 (Cobas 6000)")
    print("Instrument timeline: DxC 660i → Cobas 6000 → Cobas 8000")
    print("\nCalibration equations (converting TO 2017-2020 reference):")
    print("  2015-16 (DxC 660i):")
    print("    Given: Y(DxC 660i) = 1.150 * X(Cobas 6000) – 0.3397")
    print("    Applied: X(Cobas 6000) = (Y(DxC 660i) + 0.3397) / 1.150")
    print("    Note: Only valid for Cobas 6000 values ≤ 20 mg/L")
    print("\n  2021-23 (Cobas 8000):")
    print("    Given: Cobas 8000 = -0.08653 + 0.9672 * (Cobas 6000)")
    print("    Applied: Cobas 6000 = (Cobas 8000 + 0.08653) / 0.9672")
    print("\nCALIBRATION BY CYCLE:")
    print("  2015-16: DxC 660i → Cobas 6000 (with ≤20 mg/L limitation)")
    print("  2017-20: REFERENCE (no calibration)")
    print("  2021-23: Cobas 8000 → Cobas 6000")
    print("\n WARNING: 2015-16 calibration only valid for values ≤20 mg/L")
    print("=" * 60)

def apply_crp_calibrations(df):
    """
    Apply CRP calibrations to your NHANES dataframe
    
    Parameters:
    df: Your dataframe with columns ['SEQN', 'crp', 'CYCLE']
    
    Returns:
    Calibrated dataframe saved as crp_cal
    """
    
    # Print calibration summary
    print_crp_calibration_summary()
    print("\n")
    
    # Check for high CRP values in 2015-16 that exceed calibration limit
    if '15_16' in df['CYCLE'].values:
        high_crp_15_16 = df[(df['CYCLE'] == '15_16') & (df['crp'] > 20)]['crp']
        if len(high_crp_15_16) > 0:
            print(f"   WARNING: Found {len(high_crp_15_16)} CRP values > 20 mg/L in 2015-16 cycle")
            print(f"    Max value: {high_crp_15_16.max():.2f} mg/L")
            print(f"    Calibration equation only valid for ≤20 mg/L")
            print(f"    Consider excluding these values or using alternative approach\n")
    
    # Apply calibrations
    df_calibrated = calibrate_nhanes_crp(df)
    df_calibrated = add_crp_calibration_notes(df_calibrated)
    
    # Calculate changes for reference
    df_calibrated['crp_change'] = (df_calibrated['crp_calibrated'] - 
                                   df_calibrated['crp']).round(4)
    
    # Save calibrated results to specified dataframe name
    global crp_cal
    crp_cal = df_calibrated
    
    print("CRP CALIBRATION COMPLETE!")
    print(f"Calibrated results saved to 'crp_cal' dataframe")
    print(f"Shape: {crp_cal.shape}")
    print("\nNew columns added:")
    print("  - crp_calibrated: Calibrated CRP values (mg/L)")
    print("  - crp_calibration_equations: Notes on equations applied")
    print("  - crp_change: Change from original CRP")
    
    # Summary statistics
    print(f"\nCALIBRATION IMPACT SUMMARY:")
    for cycle in df_calibrated['CYCLE'].unique():
        cycle_data = df_calibrated[df_calibrated['CYCLE'] == cycle]
        if not cycle_data['crp_change'].isna().all():
            mean_change = cycle_data['crp_change'].mean()
            print(f"  {cycle}: Mean change = {mean_change:.4f} mg/L")
    
    return crp_cal

In [21]:
# apply calibrations
crp_cal = apply_crp_calibrations(crp_cycle)

NHANES C-REACTIVE PROTEIN CALIBRATION

CRP REFERENCE: 2017-2020 (Cobas 6000)
Instrument timeline: DxC 660i → Cobas 6000 → Cobas 8000

Calibration equations (converting TO 2017-2020 reference):
  2015-16 (DxC 660i):
    Given: Y(DxC 660i) = 1.150 * X(Cobas 6000) – 0.3397
    Applied: X(Cobas 6000) = (Y(DxC 660i) + 0.3397) / 1.150
    Note: Only valid for Cobas 6000 values ≤ 20 mg/L

  2021-23 (Cobas 8000):
    Given: Cobas 8000 = -0.08653 + 0.9672 * (Cobas 6000)
    Applied: Cobas 6000 = (Cobas 8000 + 0.08653) / 0.9672

CALIBRATION BY CYCLE:
  2015-16: DxC 660i → Cobas 6000 (with ≤20 mg/L limitation)
  2017-20: REFERENCE (no calibration)
  2021-23: Cobas 8000 → Cobas 6000



    Max value: 188.50 mg/L
    Calibration equation only valid for ≤20 mg/L
    Consider excluding these values or using alternative approach

CRP CALIBRATION COMPLETE!
Calibrated results saved to 'crp_cal' dataframe
Shape: (30192, 6)

New columns added:
  - crp_calibrated: Calibrated CRP values (mg/L)
  - crp_calib

In [22]:
# select columns
crp_cal = crp_cal[['SEQN', 'crp_calibrated']]
# rename
crp_cal = crp_cal.rename(columns={'crp_calibrated':'crp'})

# merge with ingredients dataset
wweia_ingredients = wweia_ingredients.merge(crp_cal, on='SEQN', how='left')

# cutoff to remove very high level of CRP which indicates acute phase response (infection)
wweia_ingredients = wweia_ingredients[wweia_ingredients['crp']<=10]

In [23]:
# select columns of interest
wweia_ingredients = wweia_ingredients[['SEQN', 'ingred_code', 'ingred_desc', 'Ingred_consumed_g', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity',
                 'family_pir', 'education', 'Energy', 'hypertension', 'glucose_calibrated', 'insulin_calibrated', 'crp']]

### Energy intake - exclude participants with extreme caloric intake (<500, >4500 per day)

In [24]:
# sum calories for each participant
energy = wweia_ingredients.groupby('SEQN')['Energy'].agg("sum")
energy = energy.reset_index()

# exclude participants with <500 or >4500 kcals
energy = energy[energy['Energy']>500]
energy = energy[energy['Energy']<4500]

# select participants with the range of caloric intake
wweia_ingredients_qc = wweia_ingredients[wweia_ingredients['SEQN'].isin(energy['SEQN'])] 

### Define metabolically at risk participants where BMI >= 25 and WC >= 102 for males and >= 88 for females

In [25]:
# create a copy
wweia_ingredients_qc = wweia_ingredients_qc.copy()

# create variable defining metabolically at-risk
wweia_ingredients_qc['met_risk'] = np.where(
    (wweia_ingredients_qc['BMI'] >= 25) & 
    (((wweia_ingredients_qc['Sex'] == 'Male') & (wweia_ingredients_qc['WC'] >= 102)) |
     ((wweia_ingredients_qc['Sex'] == 'Female') & (wweia_ingredients_qc['WC'] >= 88))),
    'yes', 'no'
)

# apply to the data
wweia_ingredients_qc = wweia_ingredients_qc[wweia_ingredients_qc['met_risk'] == 'yes']

### Check missing values

In [26]:
# examine NAs
print(wweia_ingredients_qc.drop_duplicates(subset='SEQN').isna().sum())

# fewer than 10% NAs for any given feature, dropNAs and proceed with complete case analysis
wweia_ingredients_qc = wweia_ingredients_qc.dropna()

# final sample size
print('\nFinal sample size for IR dataset is:', wweia_ingredients_qc.SEQN.nunique())

SEQN                     0
ingred_code              0
ingred_desc              0
Ingred_consumed_g        0
BMI                      0
WC                       0
Sex                      0
Age                      0
ethnicity                0
family_pir             841
education                0
Energy                   0
hypertension             0
glucose_calibrated    6041
insulin_calibrated    6041
crp                      0
met_risk                 0
dtype: int64

Final sample size for IR dataset is: 3585


### Calculate HOMA-IR

In [27]:
# homa-ir calculation:
wweia_ingredients_qc['homa_ir'] = wweia_ingredients_qc['insulin_calibrated'] * wweia_ingredients_qc['glucose_calibrated'] / 405

# clinical threshold to determine if insulin resistant based on homa ir is >= 2.5
wweia_ingredients_qc['ir'] = np.where(wweia_ingredients_qc['homa_ir'] >= 2.5, 1, 0)

# clean up columns
wweia_ingredients_qc = wweia_ingredients_qc.drop(columns=['insulin_calibrated', 'glucose_calibrated', 'homa_ir'])

### Create a dataframe with person specific characteristics to merge with the other dietary datasets

In [28]:
# split off demographics, covariates and outcomes to merge with other datasets
demo = wweia_ingredients_qc.drop_duplicates(subset='SEQN')

demo = demo[['SEQN', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity', 'family_pir', 'education',
       'hypertension', 'ir', 'crp']]

# convert participant IDs as int, rather than float
demo['SEQN'] = demo['SEQN'].astype(int)

### Generate the IR outcome dataset

In [29]:
# get only the participant ID and IR target outcome for saving
ir = demo[['SEQN', 'ir']]
ir = ir.copy()

# convert from 0/1 to no/yes
ir['ir'] = np.where(ir['ir']==0, 'no', 'yes')

# save the ir outcome
ir.to_csv('../../data/01/insulin_resistance/ir_target.tsv', sep='\t', index=None)

In [30]:
# drop non-demographic features for merging
demo = demo.drop(columns=['crp', 'ir'])

### WWEIA Nutrients Dataset
- merge with covariates
- match participants with ingredients dataset
- adjust intakes based on energy intake

In [31]:
# load dataset
total_nutrients = pd.read_csv('../../data/00/wweia_dataset/wweia_total_nutrients_recalls_2023.csv')

# sum caloric intake per participant
nutrient_data_calories = total_nutrients.groupby('SEQN')['Energy (kcal)'].agg("sum")

# set index to apply energy adjustment
total_nutrients = total_nutrients.set_index(['SEQN', 'Energy (kcal)'])

# apply energy adjustment to correct for differences in total enegry consumed (per 1000 kcal for all participants)
nutrient_data_adjust = total_nutrients.div(nutrient_data_calories, axis = 0, level = 0)
nutrient_data_adjust = nutrient_data_adjust.mul(1000)
nutrient_data_adjust = nutrient_data_adjust.reset_index()

# set participant ID to integer
nutrient_data_adjust['SEQN'] = nutrient_data_adjust['SEQN'].astype(int)

# merge with covariates
nutrients_ir = demo.merge(nutrient_data_adjust, on='SEQN', how='left')

# save dataset
nutrients_ir.to_csv('../../data/01/insulin_resistance/wweia_nutrients_all_features.tsv', sep='\t', index=None)

In [33]:
# load polyphenol dataset to match nutrient intake with polyphenol intake
polyphenol = pd.read_csv('../../data/01/polyphenol/NHANES_polyphenol_content_total_by_subject.csv') 

# merge nutrients and polyphenol data
nutrients_poly = polyphenol.merge(nutrient_data_adjust, on='SEQN', how='left')

# save dataset
nutrients_poly.to_csv('../../data/01/polyphenol/total_nutrients_for_polyphenols.csv', index=None)

In [34]:
# Convert nutrient data to long format for tree based analysis
nutrient_data_long = pd.melt(nutrients_ir, 
                  id_vars=['SEQN', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity', 'family_pir',
       'education', 'hypertension'],
                  value_vars=['Energy (kcal)', 'Protein (gm)',
       'Carbohydrate (gm)', 'Total sugars (gm)', 'Dietary fiber (gm)',
       'Total fat (gm)', 'Total saturated fatty acids (gm)',
       'Total monounsaturated fatty acids (gm)',
       'Total polyunsaturated fatty acids (gm)', 'Cholesterol (mg)',
       'Vitamin E as alpha-tocopherol (mg)',
       'Added alpha-tocopherol (Vitamin E) (mg)', 'Retinol (mcg)',
       'Vitamin A, RAE (mcg)', 'Alpha-carotene (mcg)', 'Beta-carotene (mcg)',
       'Beta-cryptoxanthin (mcg)', 'Lycopene (mcg)',
       'Lutein + zeaxanthin (mcg)', 'Thiamin (Vitamin B1) (mg)',
       'Riboflavin (Vitamin B2) (mg)', 'Niacin (mg)', 'Vitamin B6 (mg)',
       'Total Folate (mcg)', 'Folic acid (mcg)', 'Food folate (mcg)',
       'Folate, DFE (mcg)', 'Vitamin B12 (mcg)', 'Added vitamin B12 (mcg)',
       'Vitamin C (mg)', 'Vitamin K (mcg)', 'Calcium (mg)', 'Phosphorus (mg)',
       'Magnesium (mg)', 'Iron (mg)', 'Zinc (mg)', 'Copper (mg)',
       'Sodium (mg)', 'Potassium (mg)', 'Selenium (mcg)', 'Caffeine (mg)',
       'Theobromine (mg)', 'Alcohol (gm)', 'Moisture (gm)',
       'SFA 4:0 (Butanoic) (gm)', 'SFA 6:0 (Hexanoic) (gm)',
       'SFA 8:0 (Octanoic) (gm)', 'SFA 10:0 (Decanoic) (gm)',
       'SFA 12:0 (Dodecanoic) (gm)', 'SFA 14:0 (Tetradecanoic) (gm)',
       'SFA 16:0 (Hexadecanoic) (gm)', 'SFA 18:0 (Octadecanoic) (gm)',
       'MFA 16:1 (Hexadecenoic) (gm)', 'MFA 18:1 (Octadecenoic) (gm)',
       'MFA 20:1 (Eicosenoic) (gm)', 'MFA 22:1 (Docosenoic) (gm)',
       'PFA 18:2 (Octadecadienoic) (gm)', 'PFA 18:3 (Octadecatrienoic) (gm)',
       'PFA 18:4 (Octadecatetraenoic) (gm)',
       'PFA 20:4 (Eicosatetraenoic) (gm)', 'PFA 20:5 (Eicosapentaenoic) (gm)',
       'PFA 22:5 (Docosapentaenoic) (gm)', 'PFA 22:6 (Docosahexaenoic) (gm)'],
                  var_name='nutrient_description',
                  value_name='nutrient_intake_per_1000kcal')

print(nutrient_data_long.head())

nutrient_data_long.to_csv('../../data/01/insulin_resistance/wweia_nutrients_long.tsv', sep='\t', index=None)

    SEQN    BMI     WC     Sex   Age           ethnicity  family_pir  \
0  21165  27.45  102.6  Female  75.0  Non-Hispanic_White        1.22   
1  21313  34.82  105.3  Female  22.0  Non-Hispanic_White        0.96   
2  21534  35.53  108.5    Male  31.0    Mexican_American        1.44   
3  21768  28.61  100.0  Female  85.0      Other_Hispanic        1.24   
4  22537  36.31  108.2  Female  40.0  Non-Hispanic_Black        5.00   

                            education hypertension nutrient_description  \
0      less than high school graduate           no        Energy (kcal)   
1  high school graduate or equivalent           no        Energy (kcal)   
2                        some college           no        Energy (kcal)   
3      less than high school graduate          yes        Energy (kcal)   
4                    college graduate          yes        Energy (kcal)   

   nutrient_intake_per_1000kcal  
0                        1586.0  
1                         960.5  
2             

### WWEIA Mixed Meal Dataset
- merge with covariates
- match participants with ingredients dataset
- adjust intakes based on energy intake

In [ ]:
# load data
mixed_meals = pd.read_csv('../../data/00/wweia_dataset/wweia_foodcode_recalls_2023.csv')

# set participant ID to int
mixed_meals['SEQN'] = mixed_meals['SEQN'].astype(int)

# set foodcode ID to int
mixed_meals['foodcode'] = mixed_meals['foodcode'].astype(int)

# merge with covariates
mixed_meals_demo = demo.merge(mixed_meals, on='SEQN', how='left')

# sum calorie intake for each participant
mixed_meals_demo['total_kcal_per_person'] = mixed_meals_demo.groupby('SEQN')['foodcode_kcal'].transform('sum')

# then scale the gram intake to 1000 kcal
mixed_meals_demo['foodcode_intake_g_per_1000kcal'] = (mixed_meals_demo['foodcode_intake_g'] / mixed_meals_demo['total_kcal_per_person']) * 1000

# drop the intermediate column
mixed_meals_demo = mixed_meals_demo.drop(['total_kcal_per_person', 'foodcode_intake_g', 'foodcode_kcal'], axis=1)

# save long format
mixed_meals_demo.to_csv('../../data/01/insulin_resistance/wweia_mixed_meals_long.tsv', sep='\t', index=None)

# pivot to wide format where each food description is a column
mixed_meals_by_food_g = mixed_meals_demo.pivot(columns = 'food_description', values= ['foodcode_intake_g_per_1000kcal'], index=['SEQN', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity', 'family_pir', 'education',
       'hypertension'])

# fill the NAs with zeros (NAs occur where a person did not consume a given food, so we set that value to zero)
mixed_meals_by_food_g = mixed_meals_by_food_g.fillna(0)

# flatten index
mixed_meals_by_food_g.columns = mixed_meals_by_food_g.columns.to_flat_index()
mixed_meals_by_food_g = mixed_meals_by_food_g.reset_index()

# set index
mixed_meals_by_food_g.set_index('SEQN', inplace=True)

# remove columns that have zero sums
mixed_meals_by_food_g = mixed_meals_by_food_g.loc[:, (mixed_meals_by_food_g.sum(axis=0) != 0)]

# make the directory
os.makedirs('../../data/01/insulin_resistance/', exist_ok=True)

# save dataset
mixed_meals_by_food_g.to_csv('../../data/01/insulin_resistance/wweia_mixed_meals_wide.tsv', sep='\t', index=True)

### WWEIA FPED Dataset
- merge with covariates
- match participants with ingredients dataset
- adjust intakes based on energy intake

In [36]:
# Load FPED dataset
fped = pd.read_csv('../../data/00/wweia_dataset/wweia_fped_recalls_2023.csv')

In [37]:
# function to calculate FPED per 1000 calories for each participant
def calculate_fped_per_1000_calories(df):
    """
    Returns the FPED components per 1000 calories
    """
    # Identify FPED columns
    non_fped_cols = ['SEQN', 'foodcode', 'food_description', 'foodcode_intake_g', 
                     'foodcode_intake_dry_g', 'foodcode_kcal', 'DESCRIPTION']
    fped_cols = [col for col in df.columns if col not in non_fped_cols]
    
    # Calculate actual FPED intake per foodcode
    df_calc = df.copy()
    for fped_col in fped_cols:
        df_calc[f'{fped_col}_intake'] = df_calc[fped_col] * (df_calc['foodcode_intake_g'] / 100)
    
    # Group by participant and sum
    agg_dict = {'foodcode_kcal': 'sum'}
    agg_dict.update({f'{col}_intake': 'sum' for col in fped_cols})
    
    participant_sums = df_calc.groupby('SEQN').agg(agg_dict).reset_index()
    
    # Calculate per 1000 calories
    result_df = pd.DataFrame({'SEQN': participant_sums['SEQN'],
                             'total_kcal': participant_sums['foodcode_kcal']})
    
    for fped_col in fped_cols:
        intake_col = f'{fped_col}_intake'
        per_1000_col = f'{fped_col}_per_1000_kcal'
        result_df[per_1000_col] = (participant_sums[intake_col] / participant_sums['foodcode_kcal']) * 1000
    
    # Handle division by zero
    result_df = result_df.fillna(0)
    
    return result_df

In [38]:
# call function:
fped_per_1000 = calculate_fped_per_1000_calories(fped)

In [39]:
# remove total calories
fped_per_1000 = fped_per_1000.drop(columns='total_kcal')

# merge with covariates
fped_per_1000_demo = demo.merge(fped_per_1000, on='SEQN', how='left')

# merge with polyphenol data
fped_per_1000_poly = polyphenol.merge(fped_per_1000, on='SEQN', how='left')

# remove the polyphenol intake columns
fped_per_1000_poly.set_index('SEQN',inplace=True)
fped_per_1000_poly = fped_per_1000_poly.iloc[:, 5:]

# save datasets
fped_per_1000_demo.to_csv('../../data/01/insulin_resistance/wweia_fped.tsv', sep='\t', index=None)
fped_per_1000_poly.to_csv('../../data/01/polyphenol/fped_for_polyphenols.csv', index=True)

### Finalize WWEIA Ingredients Dataset
- adjust intakes based on energy intake
- pivot to wide format

In [40]:
# scale food intake per 1000 kcal
wweia_ingredients_qc['total_kcal_per_person'] = wweia_ingredients_qc.groupby('SEQN')['Energy'].transform('sum')

# Then scale the gram intake to 1000 kcal
wweia_ingredients_qc['ingredient_intake_g_per_1000kcal'] = (wweia_ingredients_qc['Ingred_consumed_g'] / wweia_ingredients_qc['total_kcal_per_person']) * 1000

# drop the intermediate columns
wweia_ingredients_qc = wweia_ingredients_qc.drop(['total_kcal_per_person', 'Ingred_consumed_g'], axis=1)

# save the long format dataset
wweia_ingredients_qc.drop(columns=['Energy']).to_csv('../../data/01/insulin_resistance/wweia_ingredients_long.tsv', sep='\t', index=None)

# Sum up all intake for each person-ingredient combination
wweia_ingredients_agg = wweia_ingredients_qc.groupby(['SEQN', 'ingred_desc', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity',
       'family_pir', 'education', 'hypertension'])['ingredient_intake_g_per_1000kcal'].sum().reset_index()

# Now pivot the aggregated data
ingred_by_g = wweia_ingredients_agg.pivot(columns='ingred_desc', 
                               values='ingredient_intake_g_per_1000kcal', 
                               index=['SEQN', 'BMI', 'WC', 'Sex', 'Age', 'ethnicity',
       'family_pir', 'education', 'hypertension'])

# fill NAs for ingredients not consumed by a given participant
ingred_by_g = ingred_by_g.fillna(0)

# remove columns that sum to zero
ingred_by_g = ingred_by_g.loc[:, (ingred_by_g.sum(axis=0) != 0)]

# reset index
ingred_by_g = ingred_by_g.reset_index()

# change participant ID to int
ingred_by_g['SEQN'] = ingred_by_g['SEQN'].astype(int)

# drop duplicates
ingred_by_g = ingred_by_g.drop_duplicates(subset='SEQN')

# save the wide format dataset
ingred_by_g.to_csv('../../data/01/insulin_resistance/wweia_ingredients_wide.tsv', sep='\t', index=None)

### Match the participants from the ingredients dataset with polyphenol intake data 

In [41]:
# load the ingredients data with FDA-FDD ingredient descriptions for generating the polyphenol-ingredients dataset
ingredients = pd.read_csv('../../data/00/wweia_dataset/wweia_ingredients_fda_desc_recalls_2023.csv') # already adjust per 1000 kcal per participant

In [42]:
# subset to participants with polyphenol intake data
ingredients = ingredients[ingredients['SEQN'].isin(polyphenol['SEQN'])]

In [43]:
# Sum up all intake for each person-ingredient combination
ingredients_agg = ingredients.groupby(['SEQN', 'fda_desc'])['ingredient_intake_g_per_1000kcal'].sum().reset_index()

# Now pivot the aggregated data
ingred_by_g = ingredients_agg.pivot(columns='fda_desc', 
                               values='ingredient_intake_g_per_1000kcal', 
                               index=['SEQN'])

# fill NAs for ingredients not consumed by a given participant
ingred_by_g = ingred_by_g.fillna(0)

# remove columns that sum to zero
ingred_by_g = ingred_by_g.loc[:, (ingred_by_g.sum(axis=0) != 0)]

# reset index
ingred_by_g = ingred_by_g.reset_index()

# change participant ID to int
ingred_by_g['SEQN'] = ingred_by_g['SEQN'].astype(int)

# drop duplicates
ingred_by_g = ingred_by_g.drop_duplicates(subset='SEQN')

# save polyphenol-ingredients dataset
ingred_by_g.to_csv('../../data/01/polyphenol/ingredients_for_polyphenols.tsv', sep='\t', index=None)